# Chapter 3.2: LLMEngine & AsyncLLMEngine

> **Note: This walkthrough is based on vLLM v0.5.x architecture.** The engine internals evolve rapidly; always refer to the latest source code.

## Learning Objectives

By the end of this notebook, you will:

1. Understand `LLMEngine` — the synchronous core engine that orchestrates everything
2. Know the complete initialization flow (`__init__`)
3. Understand `add_request()` — how new requests enter the system
4. Master `step()` — the main engine loop (schedule → execute → process)
5. Understand `AsyncLLMEngine` — the async wrapper for high-throughput serving
6. Know the engine output processing pipeline
7. Be able to use LLMEngine directly for custom workflows

[![Open In Colab](https://colab.research.google.com/assets/colab-badge.svg)](https://colab.research.google.com/github/hideak1/vllm_study/blob/main/notebooks/part3/chapter_3.2_llm_engine.ipynb)
[![Download Notebook](https://img.shields.io/badge/Download-Notebook-blue)](https://raw.githubusercontent.com/hideak1/vllm_study/main/notebooks/part3/chapter_3.2_llm_engine.ipynb)

**How to do the exercises:**
1. **Google Colab (Recommended)**: Click the "Open in Colab" badge above — you get your own copy with free GPU.
2. **Local Jupyter**: Clone the repo, run `./start.sh`, then open notebooks at `http://localhost:8888`.
3. **Exercises**: Look for cells marked with 🏋️ **Exercise** or **Assignment**. Fill in the `TODO` sections and run the cell to check your work.

> **Tip**: In Colab, the notebook is automatically copied to your Drive — your changes are saved there.

## 1. Overview: LLMEngine is the Heart of vLLM

The `LLMEngine` is the central orchestrator of vLLM. Every request, whether from the API server, the Python `LLM` class, or any other entry point, flows through `LLMEngine`.

```
                    ┌─────────────────────┐
                    │    LLMEngine         │
                    │                     │
   add_request() ──►│  ┌───────────────┐  │
                    │  │  Scheduler     │  │
                    │  │  (what to run) │  │
                    │  └───────┬───────┘  │
      step() ──────►│          │          │
                    │          ▼          │
                    │  ┌───────────────┐  │
                    │  │  Executor     │  │
                    │  │  (run model)  │  │
                    │  └───────┬───────┘  │
                    │          │          │
                    │          ▼          │
                    │  ┌───────────────┐  │──► RequestOutput
                    │  │  Output Proc  │  │
                    │  │  (process)    │  │
                    │  └───────────────┘  │
                    └─────────────────────┘
```

**Key insight**: `LLMEngine` is **synchronous**. Each call to `step()` does exactly one iteration of: schedule → execute → process. The `AsyncLLMEngine` wraps this in an async loop.

In [ ]:
import matplotlib.pyplot as plt
import matplotlib.patches as mpatches
import matplotlib.patheffects as pe

# ── Figure 1: Engine step() Loop Flowchart ──
BLUE = '#4A90D9'; GREEN = '#27AE60'; ORANGE = '#F39C12'; RED = '#E74C3C'; PURPLE = '#8E44AD'

fig, ax = plt.subplots(figsize=(12, 9))
ax.set_xlim(0, 12); ax.set_ylim(0, 10)
ax.axis('off'); fig.patch.set_facecolor('white')

# Nodes: (x, y, label, color, shape)
nodes = [
    (6, 9.2, 'LLMEngine.step()', BLUE),
    (6, 7.6, 'schedule()\nScheduler picks sequences\n& allocates KV blocks', BLUE),
    (6, 5.8, 'execute_model()\nWorker runs forward pass\non GPU', PURPLE),
    (6, 4.0, 'process_outputs()\nAppend tokens, check\nstopping criteria', GREEN),
    (6, 2.2, 'Return\nRequestOutput', ORANGE),
]

bw, bh = 3.4, 1.1
for x, y, label, color in nodes:
    rect = mpatches.FancyBboxPatch((x-bw/2, y-bh/2), bw, bh,
        boxstyle="round,pad=0.15", facecolor=color, edgecolor='white',
        linewidth=2, alpha=0.9)
    ax.add_patch(rect)
    ax.text(x, y, label, ha='center', va='center', fontsize=9,
            fontweight='bold', color='white')

# Arrows
akw = dict(arrowstyle='->', color='#2C3E50', lw=2)
for i in range(len(nodes)-1):
    ax.annotate('', xy=(6, nodes[i+1][1]+bh/2+0.05),
                xytext=(6, nodes[i][1]-bh/2-0.05), arrowprops=akw)

# Loop-back arrow
ax.annotate('', xy=(2.2, nodes[0][1]),
            xytext=(2.2, nodes[3][1]),
            arrowprops=dict(arrowstyle='->', color=RED, lw=2,
                           connectionstyle='arc3,rad=0'))
ax.annotate('', xy=(6-bw/2-0.05, nodes[0][1]),
            xytext=(2.2, nodes[0][1]),
            arrowprops=dict(arrowstyle='->', color=RED, lw=2))
ax.text(1.2, 5.8, 'Repeat until\nall requests\nfinished', fontsize=9,
        color=RED, fontweight='bold', ha='center',
        bbox=dict(boxstyle='round', facecolor='#FDEDEC', edgecolor=RED, alpha=0.8))

ax.set_title('LLMEngine.step() Loop', fontsize=16, fontweight='bold', pad=10)
plt.tight_layout(); plt.show()

## 2. LLMEngine.__init__() — Initialization Flow

Let's walk through exactly what happens when an LLMEngine is created.

In [ ]:
import matplotlib.pyplot as plt
import matplotlib.patches as mpatches

# ── Figure 2: Async vs Sync Engine Comparison ──
BLUE = '#4A90D9'; GREEN = '#27AE60'; ORANGE = '#F39C12'; PURPLE = '#8E44AD'

fig, (ax1, ax2) = plt.subplots(1, 2, figsize=(16, 7))

# --- Left: LLMEngine (Synchronous / Blocking) ---
ax1.set_xlim(0, 10); ax1.set_ylim(0, 8); ax1.axis('off')
ax1.set_title('LLMEngine (Synchronous)', fontsize=14, fontweight='bold')

sync_steps = [
    (5, 7.0, 'Caller thread', '#95A5A6', 3.5),
    (5, 5.5, 'add_request()', BLUE, 3.5),
    (5, 4.0, 'step()  [BLOCKS]', ORANGE, 3.5),
    (5, 2.5, 'Return output', GREEN, 3.5),
]
for x, y, label, color, w in sync_steps:
    rect = mpatches.FancyBboxPatch((x-w/2, y-0.45), w, 0.9,
        boxstyle="round,pad=0.1", facecolor=color, edgecolor='white',
        linewidth=2, alpha=0.85)
    ax1.add_patch(rect)
    ax1.text(x, y, label, ha='center', va='center', fontsize=10,
             fontweight='bold', color='white')

for i in range(len(sync_steps)-1):
    ax1.annotate('', xy=(5, sync_steps[i+1][1]+0.45),
                 xytext=(5, sync_steps[i][1]-0.45),
                 arrowprops=dict(arrowstyle='->', color='#2C3E50', lw=2))

ax1.text(5, 1.2, 'Caller is BLOCKED during step().\nUsed for offline / batch inference.',
         ha='center', fontsize=9, style='italic', color='#7F8C8D')

# --- Right: AsyncLLMEngine (Event-loop) ---
ax2.set_xlim(0, 10); ax2.set_ylim(0, 8); ax2.axis('off')
ax2.set_title('AsyncLLMEngine (Async)', fontsize=14, fontweight='bold')

ax2.add_patch(mpatches.FancyBboxPatch((0.5, 0.5), 9, 7,
    boxstyle="round,pad=0.2", facecolor='#EBF5FB', edgecolor=BLUE,
    linewidth=2, linestyle='--'))
ax2.text(5, 7.8, 'asyncio event loop', fontsize=10, ha='center',
         color=BLUE, fontweight='bold')

async_items = [
    (3, 6.0, 'generate()\ncoroutine', BLUE, 3.0),
    (7.5, 6.0, 'Background\nstep() task', ORANGE, 3.0),
    (3, 3.5, 'Yield tokens\nvia AsyncIterator', GREEN, 3.0),
    (7.5, 3.5, 'Scheduler +\nexecute_model()', PURPLE, 3.0),
]
for x, y, label, color, w in async_items:
    rect = mpatches.FancyBboxPatch((x-w/2, y-0.55), w, 1.1,
        boxstyle="round,pad=0.1", facecolor=color, edgecolor='white',
        linewidth=2, alpha=0.85)
    ax2.add_patch(rect)
    ax2.text(x, y, label, ha='center', va='center', fontsize=9,
             fontweight='bold', color='white')

ax2.annotate('', xy=(7.5, 3.5+0.55), xytext=(7.5, 6.0-0.55),
             arrowprops=dict(arrowstyle='->', color='#2C3E50', lw=2))
ax2.annotate('', xy=(3, 3.5+0.55), xytext=(3, 6.0-0.55),
             arrowprops=dict(arrowstyle='->', color='#2C3E50', lw=2))
ax2.annotate('', xy=(3+1.5, 3.5), xytext=(7.5-1.5, 3.5),
             arrowprops=dict(arrowstyle='<->', color=ORANGE, lw=1.5, linestyle='--'))

ax2.text(5, 1.5, 'Non-blocking. step() runs in background.\n'
         'Used for online serving (API server).',
         ha='center', fontsize=9, style='italic', color='#7F8C8D')

plt.tight_layout(); plt.show()

In [ ]:
# Source: vllm/engine/llm_engine.py (annotated and simplified)

init_code = '''
class LLMEngine:
    """An LLM engine that receives requests and generates texts.
    
    This is the main class for the vLLM engine. It receives requests
    from clients and generates texts from the LLM. It includes a
    tokenizer, a language model (possibly distributed across multiple
    GPUs), and GPU memory space allocated for intermediate states
    (aka KV cache).
    """
    
    def __init__(
        self,
        model_config: ModelConfig,
        cache_config: CacheConfig,
        parallel_config: ParallelConfig,
        scheduler_config: SchedulerConfig,
        device_config: DeviceConfig,
        lora_config: Optional[LoRAConfig],
        speculative_config: Optional[SpeculativeConfig],
        ...
    ) -> None:
        # ========================================
        # STEP 1: Store all configuration objects
        # ========================================
        self.model_config = model_config
        self.cache_config = cache_config
        self.parallel_config = parallel_config
        self.scheduler_config = scheduler_config
        # ... (store all configs)
        
        # ========================================
        # STEP 2: Initialize the tokenizer
        # ========================================
        # The tokenizer is used by the engine for:
        # - Tokenizing prompts (if given as strings)
        # - Detokenizing output for streaming
        # - Computing token counts
        self.tokenizer = self._init_tokenizer()
        # The tokenizer is a TokenizerGroup that handles
        # efficient tokenization (possibly in separate process)
        
        # ========================================
        # STEP 3: Create the model executor
        # ========================================
        # The executor manages GPU workers and model execution.
        # Selection depends on: device type, parallelism config, Ray usage
        self.model_executor = executor_class(
            model_config=model_config,
            cache_config=cache_config,
            parallel_config=parallel_config,
            ...
        )
        # CRITICAL: This triggers model loading and KV cache allocation!
        # Inside the executor:
        #   1. Workers are created (one per GPU)
        #   2. Model weights are loaded onto GPUs
        #   3. GPU memory is profiled
        #   4. KV cache blocks are allocated
        
        # ========================================
        # STEP 4: Initialize KV cache
        # ========================================
        self._initialize_kv_caches()
        # This determines how many KV cache blocks fit in GPU memory
        # and creates the actual cache tensors
        
        # ========================================
        # STEP 5: Create the scheduler
        # ========================================
        # One scheduler per pipeline stage
        # (typically just one for single-pipeline configs)
        self.scheduler = [
            Scheduler(
                scheduler_config=scheduler_config,
                cache_config=cache_config,
                lora_config=lora_config,
            )
            for _ in range(parallel_config.pipeline_parallel_size)
        ]
        
        # ========================================
        # STEP 6: Initialize output processor
        # ========================================
        self.output_processor = SequenceGroupOutputProcessor(
            scheduler_config=scheduler_config,
            ...
        )
        
        # ========================================
        # STEP 7: Initialize metrics and logging
        # ========================================
        self.stat_logger = StatLogger(...)
        # Prometheus metrics for monitoring
'''
print(init_code)

In [ ]:
# The factory method: from_engine_args()
# This is how LLMEngine is typically created

factory_code = '''
@classmethod
def from_engine_args(
    cls,
    engine_args: EngineArgs,
    ...
) -> "LLMEngine":
    """Creates an LLM engine from the engine arguments."""
    
    # Step 1: Convert flat EngineArgs into structured config objects
    engine_config = engine_args.create_engine_config()
    # engine_config contains: model_config, cache_config,
    # parallel_config, scheduler_config, device_config, etc.
    
    # Step 2: Determine which executor class to use
    executor_class = cls._get_executor_cls(engine_config)
    # This selects: GPUExecutor, RayGPUExecutor, etc.
    
    # Step 3: Create the engine
    engine = cls(
        **engine_config.to_dict(),
        executor_class=executor_class,
        ...
    )
    return engine
'''
print(factory_code)

### 2.1 KV Cache Initialization Deep Dive

In [ ]:
# _initialize_kv_caches() - Critical initialization step

kv_cache_init = '''
def _initialize_kv_caches(self) -> None:
    """Profiles the GPU memory usage and initializes the KV cache.
    
    The key challenge: we need to figure out how many KV cache blocks
    we can fit in GPU memory AFTER the model weights are loaded.
    We do this by actually running a forward pass and measuring memory.
    """
    
    # Step 1: Profile memory by running a dummy forward pass
    # The executor runs a forward pass with the maximum possible batch
    # to measure peak memory usage of the model.
    num_gpu_blocks, num_cpu_blocks = (
        self.model_executor.determine_num_available_blocks()
    )
    # This returns:
    #   num_gpu_blocks: how many KV cache blocks fit on GPU
    #   num_cpu_blocks: how many KV cache blocks fit on CPU (for swap)
    
    # Step 2: Validate the numbers
    if num_gpu_blocks <= 0:
        raise ValueError(
            "No available memory for the cache blocks. "
            "Try increasing gpu_memory_utilization or "
            "reducing max_model_len."
        )
    
    # Step 3: Store in cache_config
    self.cache_config.num_gpu_blocks = num_gpu_blocks
    self.cache_config.num_cpu_blocks = num_cpu_blocks
    
    # Step 4: Tell the executor to allocate the actual cache tensors
    self.model_executor.initialize_cache(
        num_gpu_blocks=num_gpu_blocks,
        num_cpu_blocks=num_cpu_blocks,
    )
    
    # IMPORTANT LOG: This is one of the first things you see at startup
    logger.info(
        f"# GPU blocks: {num_gpu_blocks}, "
        f"# CPU blocks: {num_cpu_blocks}"
    )
    # Example output: "# GPU blocks: 7680, # CPU blocks: 512"
    # With block_size=16 and 7680 blocks, you can cache:
    # 7680 * 16 = 122,880 tokens worth of KV cache
'''
print(kv_cache_init)

## 3. add_request() — How Requests Enter the System

Every new inference request must go through `add_request()` before it can be processed.

In [ ]:
# Source: vllm/engine/llm_engine.py

add_request_code = '''
def add_request(
    self,
    request_id: str,            # Unique identifier for this request
    prompt: Optional[str],       # Text prompt (or None if using token IDs)
    sampling_params: SamplingParams,  # How to sample (temperature, top_p, etc.)
    prompt_token_ids: Optional[List[int]] = None,  # Pre-tokenized prompt
    arrival_time: Optional[float] = None,  # When request arrived
    lora_request: Optional[LoRARequest] = None,
    multi_modal_data: Optional[MultiModalData] = None,
) -> None:
    """Add a request to the engine.
    
    The request is added to the scheduler\'s WAITING queue.
    It will be scheduled for execution in subsequent step() calls.
    """
    
    # Step 1: Record arrival time (for FCFS scheduling)
    if arrival_time is None:
        arrival_time = time.monotonic()
    
    # Step 2: Tokenize the prompt (if given as string)
    if prompt_token_ids is None:
        assert prompt is not None
        prompt_token_ids = self.tokenizer.encode(prompt)
    
    # Step 3: Validate the request
    # - Check prompt length vs max_model_len
    # - Validate sampling parameters
    self._validate_request(prompt_token_ids, sampling_params)
    
    # Step 4: Create the processing artifacts
    processed_inputs = self.process_model_inputs(
        request_id=request_id,
        inputs=prompt_token_ids,
        ...
    )
    
    # Step 5: Create SequenceGroup
    # A SequenceGroup contains one or more Sequences.
    # For simple generation: 1 sequence
    # For beam search: n sequences (where n = beam width)
    # For best-of-n: n sequences
    seq = Sequence(
        seq_id=next(self.seq_counter),
        prompt=prompt,
        prompt_token_ids=prompt_token_ids,
        block_size=self.cache_config.block_size,
        ...
    )
    
    seq_group = SequenceGroup(
        request_id=request_id,
        seqs=[seq],
        sampling_params=sampling_params,
        arrival_time=arrival_time,
        lora_request=lora_request,
    )
    
    # Step 6: Add to scheduler\'s waiting queue
    self.scheduler[0].add_seq_group(seq_group)
    # Now the request sits in the WAITING queue until
    # the scheduler picks it up in a future step() call.
'''
print(add_request_code)

In [ ]:
# Visualize the request flow through add_request()

import matplotlib.pyplot as plt
import matplotlib.patches as mpatches

fig, ax = plt.subplots(1, 1, figsize=(14, 6))
ax.set_xlim(0, 14)
ax.set_ylim(0, 6)
ax.axis('off')
ax.set_title('add_request() Flow', fontsize=16, fontweight='bold')

steps = [
    (1, 5, 'Prompt\n(string)', '#FFB3BA'),
    (3.5, 5, 'Tokenize', '#BAFFC9'),
    (6, 5, 'Validate', '#BAE1FF'),
    (8.5, 5, 'Create\nSequence', '#FFFFBA'),
    (11, 5, 'Create\nSeqGroup', '#E8BAFF'),
    (13, 5, 'Waiting\nQueue', '#FFD9BA'),
]

for x, y, text, color in steps:
    rect = mpatches.FancyBboxPatch((x-0.9, y-0.6), 1.8, 1.2,
                                    boxstyle="round,pad=0.1",
                                    facecolor=color, edgecolor='black')
    ax.add_patch(rect)
    ax.text(x, y, text, ha='center', va='center', fontsize=9, fontweight='bold')

# Arrows
for i in range(len(steps)-1):
    x1 = steps[i][0] + 0.9
    x2 = steps[i+1][0] - 0.9
    ax.annotate('', xy=(x2, 5), xytext=(x1, 5),
                arrowprops=dict(arrowstyle='->', color='black', lw=1.5))

# Details below
details = [
    (1, 3.5, 'User provides\ntext or token IDs'),
    (3.5, 3.5, 'self.tokenizer\n.encode(prompt)'),
    (6, 3.5, 'Check length\n< max_model_len'),
    (8.5, 3.5, 'Sequence with\nstatus=WAITING'),
    (11, 3.5, 'Groups seqs\nfor beam search'),
    (13, 3.5, 'scheduler.add_\nseq_group()'),
]

for x, y, text in details:
    ax.text(x, y, text, ha='center', va='center', fontsize=8,
            style='italic', color='#555555')
    ax.plot([x, x], [4.1, 4.4], color='gray', linestyle='--', linewidth=0.5)

plt.tight_layout()
plt.show()

## 4. step() — The Main Engine Loop

The `step()` method is the most important method in vLLM. It is called repeatedly, and each call performs one iteration of the engine loop: **schedule → execute → process**.

In [ ]:
# Source: vllm/engine/llm_engine.py

step_code = '''
def step(self) -> List[RequestOutput]:
    """Performs one decoding iteration and returns newly generated results.
    
    This is THE core method of the engine. It:
    1. Schedules which sequences to run
    2. Executes the model forward pass
    3. Processes the outputs
    
    Returns:
        A list of RequestOutput objects for requests that have
        new output in this step (including finished requests).
    """
    
    # ╔══════════════════════════════════════════╗
    # ║  PHASE 1: SCHEDULE                       ║
    # ║  CPU-side: decide what to run            ║
    # ╚══════════════════════════════════════════╝
    seq_group_metadata_list, scheduler_outputs = self.scheduler[0].schedule()
    
    # scheduler_outputs contains:
    #   - scheduled_seq_groups: list of SequenceGroups to process
    #   - blocks_to_swap_in: blocks to move from CPU → GPU
    #   - blocks_to_swap_out: blocks to move from GPU → CPU
    #   - blocks_to_copy: blocks to copy (for CoW in beam search)
    #   - num_batched_tokens: total tokens in this batch
    
    # If nothing is scheduled, return empty
    if not scheduler_outputs.is_empty():
        
        # ╔══════════════════════════════════════════╗
        # ║  PHASE 2: EXECUTE MODEL                  ║
        # ║  GPU-side: run the forward pass          ║
        # ╚══════════════════════════════════════════╝
        execute_model_req = ExecuteModelRequest(
            seq_group_metadata_list=seq_group_metadata_list,
            blocks_to_swap_in=scheduler_outputs.blocks_to_swap_in,
            blocks_to_swap_out=scheduler_outputs.blocks_to_swap_out,
            blocks_to_copy=scheduler_outputs.blocks_to_copy,
        )
        
        output = self.model_executor.execute_model(
            execute_model_req=execute_model_req
        )
        # output is a list of SamplerOutput, containing
        # the sampled tokens for each sequence
    else:
        output = []
    
    # ╔══════════════════════════════════════════╗
    # ║  PHASE 3: PROCESS OUTPUTS                ║
    # ║  CPU-side: update state, check stopping  ║
    # ╚══════════════════════════════════════════╝
    request_outputs = self._process_model_outputs(
        output, scheduler_outputs
    )
    
    return request_outputs
'''
print(step_code)

In [ ]:
# Let's visualize the three phases of step()

fig, axes = plt.subplots(1, 3, figsize=(18, 6))

# Phase 1: Schedule
ax = axes[0]
ax.set_xlim(0, 6)
ax.set_ylim(0, 8)
ax.axis('off')
ax.set_title('Phase 1: Schedule (CPU)', fontsize=14, fontweight='bold', color='blue')

schedule_steps = [
    (3, 7, 'Check waiting\nqueue', '#BAFFC9'),
    (3, 5.5, 'Check running\nqueue', '#BAFFC9'),
    (3, 4, 'Allocate/free\nKV blocks', '#BAE1FF'),
    (3, 2.5, 'Handle\npreemption', '#FFB3BA'),
    (3, 1, 'Return\nSchedulerOutputs', '#FFFFBA'),
]
for x, y, text, color in schedule_steps:
    rect = mpatches.FancyBboxPatch((x-1.2, y-0.5), 2.4, 1.0,
                                    boxstyle="round,pad=0.1",
                                    facecolor=color, edgecolor='black')
    ax.add_patch(rect)
    ax.text(x, y, text, ha='center', va='center', fontsize=9)
for i in range(len(schedule_steps)-1):
    ax.annotate('', xy=(3, schedule_steps[i+1][1]+0.5),
                xytext=(3, schedule_steps[i][1]-0.5),
                arrowprops=dict(arrowstyle='->', color='black'))

# Phase 2: Execute
ax = axes[1]
ax.set_xlim(0, 6)
ax.set_ylim(0, 8)
ax.axis('off')
ax.set_title('Phase 2: Execute (GPU)', fontsize=14, fontweight='bold', color='green')

execute_steps = [
    (3, 7, 'Prepare input\ntensors', '#BAFFC9'),
    (3, 5.5, 'Swap KV cache\nblocks', '#BAE1FF'),
    (3, 4, 'Model forward\npass', '#E8BAFF'),
    (3, 2.5, 'Sample next\ntokens', '#FFFFBA'),
    (3, 1, 'Return\nSamplerOutput', '#FFD9BA'),
]
for x, y, text, color in execute_steps:
    rect = mpatches.FancyBboxPatch((x-1.2, y-0.5), 2.4, 1.0,
                                    boxstyle="round,pad=0.1",
                                    facecolor=color, edgecolor='black')
    ax.add_patch(rect)
    ax.text(x, y, text, ha='center', va='center', fontsize=9)
for i in range(len(execute_steps)-1):
    ax.annotate('', xy=(3, execute_steps[i+1][1]+0.5),
                xytext=(3, execute_steps[i][1]-0.5),
                arrowprops=dict(arrowstyle='->', color='black'))

# Phase 3: Process
ax = axes[2]
ax.set_xlim(0, 6)
ax.set_ylim(0, 8)
ax.axis('off')
ax.set_title('Phase 3: Process (CPU)', fontsize=14, fontweight='bold', color='red')

process_steps = [
    (3, 7, 'Append tokens\nto sequences', '#BAFFC9'),
    (3, 5.5, 'Check stopping\ncriteria', '#BAE1FF'),
    (3, 4, 'Detokenize\n(for streaming)', '#E8BAFF'),
    (3, 2.5, 'Free blocks for\nfinished seqs', '#FFB3BA'),
    (3, 1, 'Return\nRequestOutputs', '#FFFFBA'),
]
for x, y, text, color in process_steps:
    rect = mpatches.FancyBboxPatch((x-1.2, y-0.5), 2.4, 1.0,
                                    boxstyle="round,pad=0.1",
                                    facecolor=color, edgecolor='black')
    ax.add_patch(rect)
    ax.text(x, y, text, ha='center', va='center', fontsize=9)
for i in range(len(process_steps)-1):
    ax.annotate('', xy=(3, process_steps[i+1][1]+0.5),
                xytext=(3, process_steps[i][1]-0.5),
                arrowprops=dict(arrowstyle='->', color='black'))

plt.tight_layout()
plt.show()

## 5. Output Processing Pipeline

After the model produces logits and tokens are sampled, the engine must process these outputs.

In [ ]:
# _process_model_outputs() detailed walkthrough

output_processing = '''
def _process_model_outputs(
    self,
    output: List[SamplerOutput],
    scheduler_outputs: SchedulerOutputs,
) -> List[RequestOutput]:
    """Process model outputs and return results."""
    
    request_outputs: List[RequestOutput] = []
    
    # For each scheduled sequence group and its output:
    for scheduled_seq_group, outputs in zip(
        scheduler_outputs.scheduled_seq_groups,
        output[0].outputs,  # SamplerOutput.outputs
    ):
        seq_group = scheduled_seq_group.seq_group
        
        # Step 1: Process the sequence group output
        # This handles:
        # - Appending the sampled token to the sequence
        # - Forking sequences for beam search
        # - Checking stopping criteria:
        #   * max_tokens reached
        #   * EOS token generated
        #   * Stop string matched
        self.output_processor.process_outputs(
            seq_group, outputs
        )
        
        # Step 2: Free blocks for finished sequences
        if seq_group.is_finished():
            self.scheduler[0].free_finished_seq_group(seq_group)
            # This returns KV cache blocks to the free pool
        
        # Step 3: Create RequestOutput
        request_output = RequestOutput.from_seq_group(seq_group)
        request_outputs.append(request_output)
    
    # Step 4: Handle aborted requests
    for seq_group in scheduler_outputs.ignored_seq_groups:
        request_output = RequestOutput.from_seq_group(seq_group)
        request_outputs.append(request_output)
    
    return request_outputs
'''
print(output_processing)

In [ ]:
# RequestOutput structure

request_output_code = '''
# Source: vllm/outputs.py

class CompletionOutput:
    """The output data of one completion."""
    index: int                    # Index of this output in the request
    text: str                     # Generated text
    token_ids: List[int]          # Generated token IDs
    cumulative_logprob: float     # Sum of log probs
    logprobs: Optional[...]       # Per-token log probabilities
    finish_reason: Optional[str]  # "stop", "length", or None
    stop_reason: Optional[...]    # Why it stopped


class RequestOutput:
    """The output of a request to the LLM."""
    request_id: str                     # Unique request ID
    prompt: Optional[str]               # The input prompt
    prompt_token_ids: List[int]          # Input token IDs
    prompt_logprobs: Optional[...]       # Prompt token log probs
    outputs: List[CompletionOutput]      # Generated completions
    finished: bool                       # Is the request done?
    
    @classmethod
    def from_seq_group(cls, seq_group: SequenceGroup) -> "RequestOutput":
        """Create a RequestOutput from a SequenceGroup."""
        # For each sequence in the group, create a CompletionOutput
        # This handles both single-sequence and beam search cases
        seqs = seq_group.get_seqs()
        outputs = []
        for seq in seqs:
            output = CompletionOutput(
                index=...,
                text=seq.output_text,
                token_ids=seq.get_output_token_ids(),
                cumulative_logprob=seq.cumulative_logprob,
                finish_reason=SequenceStatus.get_finished_reason(seq.status),
            )
            outputs.append(output)
        
        return cls(
            request_id=seq_group.request_id,
            prompt=seq_group.prompt,
            prompt_token_ids=seq_group.prompt_token_ids,
            outputs=outputs,
            finished=seq_group.is_finished(),
        )
'''
print(request_output_code)

## 6. AsyncLLMEngine — Async Wrapper for Serving

The `AsyncLLMEngine` wraps `LLMEngine` for use with async/await patterns, which is essential for high-throughput API serving.

In [ ]:
# Source: vllm/engine/async_llm_engine.py (annotated)

async_engine_code = '''
class AsyncLLMEngine:
    """An async wrapper for LLMEngine.
    
    This class wraps LLMEngine and provides an async interface.
    It runs the engine loop in the background, continuously calling
    step() and distributing outputs to waiting callers.
    """
    
    def __init__(self, *args, **kwargs):
        # Create the underlying synchronous engine
        self.engine = LLMEngine(*args, **kwargs)
        
        # Request tracking
        self._request_tracker = RequestTracker()
        # Maps request_id → asyncio.Queue for output delivery
        # When a request is added, an output queue is created.
        # The background loop pushes outputs to this queue.
        # The API handler awaits this queue.
        
        # Background task
        self._background_loop_task: Optional[asyncio.Task] = None
    
    async def add_request(
        self,
        request_id: str,
        prompt: str,
        sampling_params: SamplingParams,
        ...
    ) -> AsyncStream:
        """Add a request and return an async stream of outputs."""
        
        # Create an output stream (async generator)
        stream = self._request_tracker.add_request(
            request_id, ...)
        
        # The actual engine.add_request() is called in the
        # background loop to avoid threading issues
        
        return stream
    
    async def generate(
        self,
        prompt: str,
        sampling_params: SamplingParams,
        request_id: str,
        ...
    ) -> AsyncIterator[RequestOutput]:
        """Generate completions for a request.
        
        Yields RequestOutput objects as they become available.
        The final RequestOutput has finished=True.
        """
        stream = await self.add_request(
            request_id, prompt, sampling_params, ...)
        
        # Yield outputs as they arrive
        async for request_output in stream:
            yield request_output
'''
print(async_engine_code)

In [ ]:
# The background loop - the heart of AsyncLLMEngine

background_loop = '''
async def _run_engine_loop(self):
    """The background loop that drives the engine.
    
    This runs continuously, calling engine.step() and
    distributing outputs to waiting callers.
    """
    while True:
        # Step 1: Process new/aborted requests
        # Check for new requests that were added via add_request()
        # and for requests that should be aborted.
        new_requests, aborted_requests = (
            self._request_tracker.get_new_and_aborted_requests()
        )
        
        for new_request in new_requests:
            self.engine.add_request(**new_request)
        
        for request_id in aborted_requests:
            self.engine.abort_request(request_id)
        
        # Step 2: Check if there\'s work to do
        if not self.engine.has_unfinished_requests():
            # No requests to process, wait for new ones
            await self._request_tracker.wait_for_new_requests()
            continue
        
        # Step 3: Run one step of the engine
        request_outputs = await self._engine_step()
        # Note: _engine_step() may run in an executor (thread pool)
        # to avoid blocking the event loop
        
        # Step 4: Distribute outputs to waiting callers
        for request_output in request_outputs:
            self._request_tracker.process_request_output(
                request_output
            )
            # This pushes the output to the appropriate
            # async queue, which the API handler is awaiting.
            # For streaming: each intermediate output is yielded.
            # For non-streaming: only the final output matters.
'''
print(background_loop)

### 6.1 Request Tracking Architecture

In [ ]:
request_tracking = """
Request Tracking in AsyncLLMEngine
════════════════════════════════════

    API Handler (coroutine 1)          Background Loop (coroutine 2)
    ─────────────────────────          ──────────────────────────────
    
    1. User sends request
    2. add_request("req-1", ...)
       └── Creates AsyncStream
       └── Adds to new_requests queue ──► Picks up new_requests
                                          └── engine.add_request()
    3. async for output in stream:
       (waiting...)                       engine.step()
                                          └── Process outputs
                                          └── Push to stream ──────► receives output!
       yield output (intermediate)        
       (waiting...)                       engine.step()
                                          └── Push to stream ──────► receives output!
       yield output (final, finished=True)
    4. Return response to user

Key Design Decisions:
  - The background loop owns the LLMEngine (thread-safety)
  - Communication is via asyncio queues (lock-free for coroutines)
  - New requests are batched before being added to the engine
  - Aborted requests are cleaned up in the background loop
"""
print(request_tracking)

## 7. has_unfinished_requests() and abort_request()

Two important utility methods that control the engine's lifecycle.

In [ ]:
utility_methods = '''
def has_unfinished_requests(self) -> bool:
    """Returns True if there are unfinished requests.
    
    This checks all three queues:
    - waiting: requests not yet started
    - running: requests currently being processed  
    - swapped: requests swapped to CPU
    
    The LLM.generate() method uses this to know when to stop:
        while engine.has_unfinished_requests():
            engine.step()
    """
    return self.scheduler[0].has_unfinished_seqs()


def abort_request(self, request_id: Union[str, Iterable[str]]) -> None:
    """Aborts a request with the given ID.
    
    Used when:
    - Client disconnects during streaming
    - Request times out
    - Server decides to cancel a request
    
    This removes the sequence group from whatever queue it is in
    and frees its KV cache blocks.
    """
    # Normalize to a set of request IDs
    if isinstance(request_id, str):
        request_id = {request_id}
    
    # Remove from scheduler and free resources
    self.scheduler[0].abort_seq_group(request_id)
    # The scheduler will:
    # 1. Find the seq_group in waiting/running/swapped queues
    # 2. Set all sequences to FINISHED_ABORTED status
    # 3. Free all allocated KV cache blocks
'''
print(utility_methods)

## 8. Demo: Using LLMEngine Directly

While most users interact with vLLM through the `LLM` class or API server, you can use `LLMEngine` directly for maximum control.

In [ ]:
# Demo: Direct LLMEngine usage (conceptual - requires GPU)

demo_code = '''
from vllm import EngineArgs, LLMEngine, SamplingParams

# Step 1: Create engine arguments
engine_args = EngineArgs(
    model="facebook/opt-125m",   # Small model for demo
    max_model_len=256,
    gpu_memory_utilization=0.5,
    enforce_eager=True,  # Easier debugging
)

# Step 2: Create the engine
engine = LLMEngine.from_engine_args(engine_args)

# Step 3: Define sampling parameters
sampling_params = SamplingParams(
    temperature=0.8,
    top_p=0.95,
    max_tokens=50,
)

# Step 4: Add requests
prompts = [
    "The future of AI is",
    "Once upon a time",
    "Python is a",
]

for i, prompt in enumerate(prompts):
    engine.add_request(
        request_id=f"req-{i}",
        prompt=prompt,
        sampling_params=sampling_params,
    )

# Step 5: Run the engine loop manually
finished_requests = {}
step_count = 0

while engine.has_unfinished_requests():
    step_count += 1
    request_outputs = engine.step()
    
    for output in request_outputs:
        if output.finished:
            finished_requests[output.request_id] = output
            print(f"[Step {step_count}] Request {output.request_id} finished!")
            print(f"  Prompt: {output.prompt}")
            print(f"  Output: {output.outputs[0].text}")
            print(f"  Tokens: {len(output.outputs[0].token_ids)}")
            print()

print(f"Total steps: {step_count}")
print(f"Finished requests: {len(finished_requests)}")
'''
print(demo_code)

In [ ]:
# Expected output pattern:

expected_output = """
Expected Output:
═══════════════

[Step 24] Request req-2 finished!
  Prompt: Python is a
  Output:  versatile programming language that is widely used for...
  Tokens: 50

[Step 31] Request req-0 finished!
  Prompt: The future of AI is
  Output:  looking increasingly bright as new advances in machine...
  Tokens: 50

[Step 35] Request req-1 finished!
  Prompt: Once upon a time
  Output:  in a land far away, there lived a young princess who...
  Tokens: 50

Total steps: 35
Finished requests: 3

Key Observations:
- Requests finish at different steps (different prefill lengths)
- All 3 requests were batched together by the scheduler
- Each step generates 1 token per running sequence (decode phase)
- Step 1 was the prefill step (processes all prompt tokens at once)
"""
print(expected_output)

## 9. Advanced: Step-by-Step Tracing

Let's trace what happens in each step for a concrete example.

In [ ]:
# Detailed step trace for a single request

step_trace = """
Request: "Hello world" → max_tokens=3
Assume: "Hello world" tokenizes to [15496, 995] (2 tokens)
Block size: 16
═══════════════════════════════════════════════════════════════════

add_request():
  - Creates Sequence(tokens=[15496, 995], status=WAITING)
  - Creates SequenceGroup with this sequence
  - Adds to scheduler waiting queue

step() #1 — PREFILL:
  Schedule:
    - Moves seq_group from WAITING → RUNNING
    - Allocates 1 physical block (block 0) for KV cache
    - seq_group_metadata: is_prompt=True, token_ids=[15496, 995]
  Execute:
    - Forward pass processes BOTH tokens simultaneously
    - KV cache for positions 0,1 written to block 0
    - Logits computed for position 1 (last token)
    - Sampler produces token, e.g., token_id=318 ("is")
  Process:
    - Appends 318 to sequence: [15496, 995, 318]
    - Check stopping: 1/3 tokens generated, not done
    - Output: RequestOutput(finished=False)

step() #2 — DECODE:
  Schedule:
    - seq_group stays in RUNNING
    - No new blocks needed (block 0 still has space for 14 more tokens)
    - seq_group_metadata: is_prompt=False, token_ids=[318]
  Execute:
    - Forward pass processes ONLY the new token (318)
    - Reads KV cache from block 0 (positions 0,1)
    - Writes KV cache for position 2 to block 0
    - Logits → Sampler produces token, e.g., 257 ("a")
  Process:
    - Appends 257: [15496, 995, 318, 257]
    - Check stopping: 2/3 tokens, not done

step() #3 — DECODE:
  Schedule:
    - Same as step 2
    - seq_group_metadata: is_prompt=False, token_ids=[257]
  Execute:
    - Forward pass: 1 token
    - Reads KV cache positions 0,1,2
    - Writes KV cache position 3
    - Sampler produces token, e.g., 1332 ("great")
  Process:
    - Appends 1332: [15496, 995, 318, 257, 1332]
    - Check stopping: 3/3 tokens = MAX TOKENS REACHED!
    - Status: RUNNING → FINISHED_LENGTH_CAPPED
    - Free block 0 back to free pool
    - Output: RequestOutput(finished=True, text="is a great")

Result: "Hello world" → "Hello world is a great"
"""
print(step_trace)

## 10. Engine Metrics and Monitoring

LLMEngine tracks various metrics for monitoring performance.

In [ ]:
metrics_info = """
=== Engine Metrics ===
Source: vllm/engine/metrics.py

Key Prometheus Metrics:

1. Throughput Metrics:
   - vllm:num_requests_running     # Currently running requests
   - vllm:num_requests_waiting     # Requests in waiting queue
   - vllm:num_requests_swapped     # Requests swapped to CPU
   - vllm:num_generation_tokens    # Total generated tokens
   - vllm:num_prompt_tokens        # Total prompt tokens processed

2. Latency Metrics:
   - vllm:time_to_first_token      # Time from request to first token (TTFT)
   - vllm:time_per_output_token    # Time per output token (TPOT)
   - vllm:e2e_request_latency      # End-to-end request latency

3. Cache Metrics:
   - vllm:gpu_cache_usage_perc     # GPU KV cache utilization %
   - vllm:cpu_cache_usage_perc     # CPU swap cache utilization %
   - vllm:num_preemptions          # Number of preemption events

4. Batch Metrics:
   - vllm:num_batched_tokens       # Tokens per batch
   - vllm:num_requests_in_batch    # Requests per batch

These are exposed via a /metrics endpoint when running the API server.
"""
print(metrics_info)

In [ ]:
# do_log_stats() — periodic logging in LLMEngine

log_stats_code = '''
def do_log_stats(self) -> None:
    """Called periodically to log engine statistics."""
    
    # Collect stats from scheduler
    num_running = self.scheduler[0].get_num_running_seqs()
    num_waiting = self.scheduler[0].get_num_waiting_seqs()
    num_swapped = self.scheduler[0].get_num_swapped_seqs()
    
    # Collect cache utilization
    gpu_cache_usage = self.scheduler[0].get_gpu_cache_usage()
    cpu_cache_usage = self.scheduler[0].get_cpu_cache_usage()
    
    # Log
    logger.info(
        f"Running: {num_running} reqs, "
        f"Waiting: {num_waiting} reqs, "
        f"GPU KV cache usage: {gpu_cache_usage:.1%}, "
        f"CPU KV cache usage: {cpu_cache_usage:.1%}"
    )
    
    # Also updates Prometheus metrics
    self.stat_logger.log(stats)
'''
print(log_stats_code)

## 11. Streaming vs Non-Streaming Output

Understanding how streaming works through the engine is important.

In [ ]:
streaming_explanation = """
Streaming vs Non-Streaming in LLMEngine
═════════════════════════════════════════

NON-STREAMING (LLM.generate()):
  1. Add all requests
  2. Call step() repeatedly until all finished
  3. Return only the FINAL RequestOutput for each request
  
  # The LLM class implements this:
  outputs = []
  while engine.has_unfinished_requests():
      step_outputs = engine.step()
      for output in step_outputs:
          if output.finished:
              outputs.append(output)
  return outputs

STREAMING (AsyncLLMEngine.generate()):
  1. Add request
  2. Each step() produces an intermediate RequestOutput
  3. EVERY intermediate output is yielded to the caller
  4. The caller (API server) sends each as an SSE event
  
  # The API server does:
  async for output in engine.generate(prompt, params, req_id):
      # output.outputs[0].text grows with each step
      # Step 1: "The"
      # Step 2: "The future"
      # Step 3: "The future of"
      # ...
      yield create_sse_event(output)  # Send to client

KEY DIFFERENCE:
  The engine itself ALWAYS produces outputs every step.
  The difference is whether the CALLER looks at intermediate outputs
  or waits for the final one.
"""
print(streaming_explanation)

## 12. Error Handling in the Engine

In [ ]:
error_handling = """
Error Handling in LLMEngine
═══════════════════════════

1. PROMPT TOO LONG:
   - Checked in add_request() → prompt_token_ids length vs max_model_len
   - Raises ValueError: "prompt too long"
   - Or: sequence is set to FINISHED_IGNORED and returned immediately

2. OUT OF MEMORY (KV Cache):
   - Handled by the scheduler through PREEMPTION
   - Running sequences are either:
     a. RECOMPUTE: evicted from GPU, will be recomputed later
     b. SWAP: KV cache moved to CPU, swapped back later
   - The engine does NOT crash on OOM - it degrades gracefully

3. GPU ERRORS:
   - CUDA errors during forward pass are caught by the executor
   - The engine may attempt to recover or shut down gracefully
   - AsyncLLMEngine catches errors and reports to all waiting requests

4. ABORT HANDLING:
   - When a client disconnects, abort_request() is called
   - Resources (KV blocks) are freed immediately
   - The sequence is removed from all queues

5. TIMEOUT:
   - The API server has configurable request timeouts
   - On timeout: abort_request() is called

AsyncLLMEngine Error Propagation:
   - If the background loop crashes, ALL pending requests get the error
   - Each waiting async generator receives the exception
   - The engine can optionally restart the loop
"""
print(error_handling)

## 13. Performance Considerations

In [ ]:
performance_notes = """
Performance Considerations for LLMEngine
══════════════════════════════════════════

1. SCHEDULING OVERHEAD:
   - The scheduler runs on CPU, so it's a potential bottleneck
   - With many sequences, schedule() can take milliseconds
   - Solution: efficient data structures, limit max_num_seqs

2. TOKENIZATION OVERHEAD:
   - Tokenization runs on CPU (HuggingFace tokenizers)
   - For long prompts, this can be significant
   - Solution: pre-tokenize if possible, or use TokenizerGroup
     with dedicated tokenizer processes

3. OUTPUT PROCESSING OVERHEAD:
   - Detokenization for streaming adds CPU work
   - Solution: incremental detokenization (only new tokens)

4. STEP() CADENCE:
   - Each step() call blocks until the GPU forward pass completes
   - GPU utilization depends on batch size
   - Small batches = GPU underutilized
   - Large batches = higher latency per step
   - Optimal: balance batch size via scheduler config

5. ASYNC OVERHEAD:
   - AsyncLLMEngine adds async queue overhead
   - The engine step may run in a thread pool (asyncio.to_thread)
   - This allows the event loop to handle HTTP requests concurrently
"""
print(performance_notes)

## Exercises

### Exercise 1: Trace a Request Through the Engine

Given the following scenario:
- Model: 7B parameter LLaMA
- Prompt: "Write a poem about the ocean" (8 tokens)
- SamplingParams: temperature=0.7, max_tokens=20
- Block size: 16

Trace the request through the engine, specifying:
1. What happens in add_request()?
2. What happens in each step() call? (at least first 3 steps)
3. How many blocks are allocated?
4. When does the request finish?

In [ ]:
# Exercise 1 Solution

exercise1_solution = """
EXERCISE 1 SOLUTION
════════════════════

add_request():
  - Tokenize "Write a poem about the ocean" → 8 token IDs
  - Validate: 8 < max_model_len ✓
  - Create Sequence(tokens=[t1..t8], status=WAITING)
  - Create SequenceGroup with 1 sequence
  - Add to scheduler waiting queue

step() #1 (PREFILL):
  Schedule:
    - Move seq_group: WAITING → RUNNING
    - Allocate 1 block (8 tokens < 16 = block_size)
    - is_prompt=True, all 8 tokens
  Execute:
    - Forward pass: 8 tokens simultaneously
    - Write KV cache: positions 0-7 in block 0
    - Logits at position 7 → sample with temp=0.7 → token_9
  Process:
    - Append token_9: now 9 tokens total
    - 1/20 generated, not finished

step() #2 (DECODE):
  Schedule: no changes, still 1 block
  Execute:
    - Forward: 1 token (token_9)
    - Read KV: positions 0-8, write position 9 in block 0
    - Sample → token_10
  Process: append token_10, 2/20

step() #3 (DECODE):
  Same pattern: 1 token forward, 3/20

... continues ...

step() #8 (DECODE):
  Schedule:
    - sequence now has 16 tokens = block 0 is full!
    - Allocate block 1 for token 17
  Execute: forward pass with 2 blocks in block table

step() #20 (DECODE):
  Process:
    - 20/20 tokens generated
    - Status: FINISHED_LENGTH_CAPPED
    - Free blocks 0 and 1
    - Return final RequestOutput

Total: 1 prefill step + 19 decode steps = 20 steps
Blocks used: 2 (for 28 tokens: 8 prompt + 20 generated)
"""
print(exercise1_solution)

### Exercise 2: Implement a Simplified Engine Loop

Write Python code that simulates the core engine loop (schedule → execute → process) without actually running a model.

In [ ]:
# Exercise 2: Simplified engine loop simulation

import time
import random
from dataclasses import dataclass, field
from typing import List, Optional, Dict
from enum import Enum

class SeqStatus(Enum):
    WAITING = "waiting"
    RUNNING = "running"
    FINISHED = "finished"

@dataclass
class SimpleSequence:
    seq_id: int
    prompt_tokens: List[int]
    generated_tokens: List[int] = field(default_factory=list)
    max_tokens: int = 10
    status: SeqStatus = SeqStatus.WAITING
    
    @property
    def total_tokens(self):
        return len(self.prompt_tokens) + len(self.generated_tokens)
    
    @property
    def is_prefill(self):
        return len(self.generated_tokens) == 0

@dataclass
class SimpleSchedulerOutput:
    scheduled_seqs: List[SimpleSequence]
    is_prefill: Dict[int, bool]  # seq_id → is_prefill

class SimpleScheduler:
    def __init__(self, max_num_seqs: int = 4):
        self.waiting: List[SimpleSequence] = []
        self.running: List[SimpleSequence] = []
        self.max_num_seqs = max_num_seqs
    
    def add_seq(self, seq: SimpleSequence):
        self.waiting.append(seq)
    
    def schedule(self) -> SimpleSchedulerOutput:
        """Simple FCFS scheduler."""
        is_prefill = {}
        
        # Move waiting → running (up to max_num_seqs)
        while self.waiting and len(self.running) < self.max_num_seqs:
            seq = self.waiting.pop(0)
            seq.status = SeqStatus.RUNNING
            self.running.append(seq)
        
        # Mark prefill status
        for seq in self.running:
            is_prefill[seq.seq_id] = seq.is_prefill
        
        return SimpleSchedulerOutput(
            scheduled_seqs=list(self.running),
            is_prefill=is_prefill,
        )
    
    def free_finished(self):
        """Remove finished sequences."""
        self.running = [s for s in self.running if s.status != SeqStatus.FINISHED]
    
    def has_unfinished(self) -> bool:
        return len(self.waiting) > 0 or len(self.running) > 0

class SimpleEngine:
    def __init__(self):
        self.scheduler = SimpleScheduler(max_num_seqs=4)
        self.seq_counter = 0
        self.vocab_size = 1000  # Fake vocabulary
    
    def add_request(self, prompt_tokens: List[int], max_tokens: int = 10):
        seq = SimpleSequence(
            seq_id=self.seq_counter,
            prompt_tokens=prompt_tokens,
            max_tokens=max_tokens,
        )
        self.seq_counter += 1
        self.scheduler.add_seq(seq)
        return seq.seq_id
    
    def step(self) -> List[SimpleSequence]:
        """One step: schedule → execute → process."""
        
        # Phase 1: Schedule
        sched_output = self.scheduler.schedule()
        
        if not sched_output.scheduled_seqs:
            return []
        
        # Phase 2: Execute (simulate model forward pass)
        for seq in sched_output.scheduled_seqs:
            if sched_output.is_prefill[seq.seq_id]:
                # Prefill: process all prompt tokens, generate 1
                new_token = random.randint(0, self.vocab_size - 1)
            else:
                # Decode: process 1 token, generate 1
                new_token = random.randint(0, self.vocab_size - 1)
            seq.generated_tokens.append(new_token)
        
        # Phase 3: Process outputs
        finished = []
        for seq in sched_output.scheduled_seqs:
            if len(seq.generated_tokens) >= seq.max_tokens:
                seq.status = SeqStatus.FINISHED
                finished.append(seq)
        
        self.scheduler.free_finished()
        return finished

# Run the simulation
engine = SimpleEngine()

# Add requests with different prompt lengths
engine.add_request([1, 2, 3], max_tokens=5)       # Short prompt
engine.add_request([10, 20, 30, 40, 50], max_tokens=3)  # Longer prompt
engine.add_request([100], max_tokens=8)            # Single token prompt

step_num = 0
while engine.scheduler.has_unfinished():
    step_num += 1
    finished = engine.step()
    
    running = len(engine.scheduler.running)
    waiting = len(engine.scheduler.waiting)
    
    status = f"Step {step_num}: running={running}, waiting={waiting}"
    if finished:
        for seq in finished:
            status += f" | Seq {seq.seq_id} FINISHED (generated {len(seq.generated_tokens)} tokens)"
    print(status)

print(f"\nTotal steps: {step_num}")

### Exercise 3: Compare Sync vs Async Patterns

Explain the key differences between using LLMEngine (sync) and AsyncLLMEngine (async). When would you use each?

In [ ]:
# Exercise 3 Solution

sync_vs_async = """
SYNC (LLMEngine) vs ASYNC (AsyncLLMEngine)
════════════════════════════════════════════

LLMEngine (Synchronous):
  USE WHEN:
  - Batch offline inference (process N prompts, get N outputs)
  - Scripts and notebooks
  - Simple applications without concurrent requests
  - Testing and debugging
  
  PROS:
  - Simpler mental model
  - Direct control over step() timing
  - Easier to debug (no async complexity)
  
  CONS:
  - Blocks the calling thread
  - Cannot handle concurrent HTTP requests efficiently
  - Cannot accept new requests while processing

AsyncLLMEngine (Asynchronous):
  USE WHEN:
  - API serving (handling many concurrent HTTP requests)
  - Streaming responses
  - Production deployments
  - When new requests arrive while others are being processed
  
  PROS:
  - Handles thousands of concurrent requests
  - Non-blocking: the event loop stays responsive
  - Native streaming support
  - New requests can be added while others are processing
  
  CONS:
  - More complex (asyncio, background tasks)
  - Harder to debug
  - Slight overhead from async queues

SUMMARY:
  Offline batch inference → LLMEngine (via LLM class)
  Online serving         → AsyncLLMEngine (via API server)
"""
print(sync_vs_async)

## Summary

In this notebook, we covered:

1. **LLMEngine.__init__()**: Creates tokenizer, executor (which loads the model), scheduler, and KV cache
2. **add_request()**: Tokenizes prompt, creates Sequence/SequenceGroup, adds to scheduler's waiting queue
3. **step()**: The three-phase loop — Schedule (what to run) → Execute (GPU forward pass) → Process (update state, check stopping)
4. **AsyncLLMEngine**: Background loop continuously calls step(), distributes outputs via async queues
5. **Output Processing**: Appends tokens, checks stopping criteria, creates RequestOutput objects
6. **Streaming**: Every step() produces output; streaming yields every intermediate result

**Key Takeaway**: LLMEngine.step() is the heartbeat of vLLM. Understanding this single method gives you the mental model for how the entire system works.

**Next**: In Chapter 3.3, we'll dive deep into the Scheduler — the brain that decides which sequences to run.